In [ ]:
import sys

# Specify the path to your package or module
# package_path = 'PATH/TO/site-packages'

# Add this path to the list of paths to search for modules
# sys.path.append(package_path)

import numpy as np
import matplotlib.pyplot as plt

from concurrent.futures import ProcessPoolExecutor, as_completed
import argparse
from tqdm import tqdm

import multivariate
import matplotlib

def get_MSE(est,ground_truth):
    mse = np.full(est.shape[1], np.nan)
    for iter_i in range(est.shape[1]):
        err = est[:,iter_i] - ground_truth[:,iter_i]
        mse[iter_i] = np.sum(err** 2)/len(err)
    return mse

def get_coverage(est, ground_truth, W, alpha): #(1 − α)% confidence band: if 90% band then alpha = 0.1
    ## derive W for band: (200, 20, 100)
    W_abs = np.abs(W)
    max_pairwise = np.nan
    T_ij = np.full(W.shape[2], np.nan)
    for iter_1 in range(W.shape[2]):
        T_ij[iter_1] = np.max(W[: , :,iter_1])
    max_abs = np.quantile(T_ij, 1 - alpha / 2)
#     print(max_abs)
    
    ratio = np.full(est.shape[1], np.nan)
    # Loop through each column (assumed to be different estimates)
    for iter_i in range(est.shape[1]):
        # Calculate the lower and upper quantiles for the current estimates
        lower_bound = est[:, iter_i] - max_abs
        upper_bound = est[:, iter_i] + max_abs
        
        # Check if the ground truth falls within the quantile interval
        coverage = (ground_truth[:, iter_i] >= lower_bound) & (ground_truth[:,iter_i] <= upper_bound)
        
        # Calculate the coverage probability
        ratio[iter_i] = np.mean(coverage)
    
    return ratio,max_abs

def pairwise_statistics(W, alpha):
    max_pairwise = np.nan
    ## Now we should consider two iterables
    ## Still save all the sampled max statistics
#     for iter_0 in range(W.shape[0]):
        ## slot for TS among all iterations
    T_ij = np.full(W.shape[2], np.nan)
    for iter_1 in range(W.shape[2]):
        TS = np.full(int(W.shape[1] * (W.shape[1] - 1) / 2), np.nan)
        sum = 0
        for iter_i in range(W.shape[1] - 1):
            for iter_j in range(iter_i + 1, W.shape[1]):
                TS[sum] = np.max(W[: , iter_i ,iter_1] - W[: , iter_j ,iter_1])
                sum += 1
        T_ij[iter_1] = np.max(TS)
    max_pairwise = np.quantile(T_ij, 1 - alpha / 2)
    return max_pairwise

def pairwise_statistics_pairw(W, alpha):
# W_{ij} = \sup_{\xb \in \Omega} \big[W_i(\xb) - W_j(\xb) \big].
#     max_pairwise = np.nan
    TS = np.full((W.shape[1], W.shape[1]), np.nan)
    for iter_i in range(W.shape[1]):
        for iter_j in range(W.shape[1]):
            T_ij = np.full(W.shape[2], np.nan)
            for iter_1 in range(W.shape[2]):
                T_ij[iter_1] = np.max(W[: , iter_i ,iter_1] - W[: , iter_j ,iter_1])
            TS[iter_i, iter_j] = np.quantile(T_ij, 1 - alpha / 2)
#     max_pairwise = np.quantile(T_ij, 1 - alpha / 2)
    return TS

def test_statistics(est, dim, time_num=200, filt=False):
# T_{ij} = \inf_{\xb \in \Omega} \sqrt{h^d \Xi} \cdot (\widehat{\theta}_i(\xb) - \widehat{\theta}_j(\xb)).
    if filt:
        T = generate_grid(0, 1, dim, time_num)
        mask = (T > 0.2) & (T < 0.8)
        # Use np.all() to filter rows where all conditions are true
        filtered_T = T[np.all(mask, axis=1)]
        indices = np.where(np.all(mask, axis=1))[0]
        est = est[indices,:]
        
    T = np.zeros((est.shape[1], est.shape[1]))
    for iter_i in range(est.shape[1]):
        for iter_j in range(est.shape[1]):
            T[iter_i, iter_j] = np.min(est[: , iter_i] - est[: , iter_j])
    return T

def pairwise_statistics_topk(W, alpha):
# W^{(i)} = \sup_{j \neq i, \xb \in \Omega} \big[W_i(\xb) - W_j(\xb) \big].
    TS = np.full(W.shape[1], np.nan)
    for iter_i in range(W.shape[1]):
        T_ij = np.full(W.shape[2], np.nan)
        for iter_1 in range(W.shape[2]):
            tmp = np.full(W.shape[1], np.nan)
            for iter_j in range(W.shape[1]):
                tmp[iter_j] = np.max(W[: , iter_i ,iter_1] - W[: , iter_j ,iter_1])
                
            # remove the effect of iter_1 from the max
            tmp[iter_i] = np.min(tmp)
            T_ij[iter_1] = np.max(tmp)
            
        TS[iter_i] = np.quantile(T_ij, 1 - alpha / 2)
#     max_pairwise = np.quantile(T_ij, 1 - alpha / 2)
    return TS

def test_statistics_topk(est, dim, time_num, filt, K):
# T_{i} = \inf_{\xb \in \Omega} \sqrt{h^d \Xi} \cdot (\widehat{\theta}_i(\xb) - \widehat{\theta}_{(K+1)}(\xb)).
    if filt:
        T = generate_grid(0, 1, dim, time_num)
        mask = (T > 0.2) & (T < 0.8)
        # Use np.all() to filter rows where all conditions are true
        filtered_T = T[np.all(mask, axis=1)]
        indices = np.where(np.all(mask, axis=1))[0]
        est = est[indices,:]
        
    T = np.zeros(est.shape[1])
    n = est.shape[1]
    est_topk = np.sort(est, axis=1)[:,n-K-1]
    for iter_i in range(est.shape[1]):
        T[iter_i] = np.min(est[: , iter_i] - est_topk)
    return T

def pairwise_test(T, quantile):
    ## type-I
    TS = np.full(int(T.shape[1] * (T.shape[1] - 1) / 2), False)
    q = quantile
    sum = 0
    for iter_i in range(T.shape[1] - 1):
                    for iter_j in range(iter_i + 1, T.shape[1]):
                        TS[sum] = (T[iter_i, iter_j] > q[iter_i, iter_j])
                        sum += 1
    ratio1 = np.sum(TS)/len(TS)

    ## power
    TS = np.full(int(T.shape[1] * (T.shape[1] - 1) / 2), False)
    sum = 0
    for iter_i in range(T.shape[1] - 1):
                    for iter_j in range(iter_i + 1, T.shape[1]):
                        TS[sum] = (T[iter_j, iter_i] > q[iter_j, iter_i])
                        sum += 1
    ratio2 = np.sum(TS)/len(TS)
    ratio = [ratio1, ratio2]
    return ratio

def topk_test(T, quantile, K):
    ## type-I
    TS = np.full(len(T), False)
    q = quantile
    sum = 0
    for iter_i in range(K+1,len(T)):
#         print(iter_i)
        TS[sum] = (T[iter_i] > q[iter_i])
        sum += 1
    ratio1 = np.sum(TS)/len(TS)
    
    ## power
    ratio2 = 0
    ratio = [ratio1, ratio2]
    return ratio

def global_test(T, quantile):
    ## for each iter
#     ratio_all = np.full(T.shape[0], np.nan)
#     for iter_0 in range(T.shape[0]):
    TS = np.full(int(T.shape[1] * (T.shape[1] - 1) / 2), False)
    q = quantile
    sum = 0
    for iter_i in range(T.shape[1] - 1):
                    for iter_j in range(iter_i + 1, T.shape[1]):
                        TS[sum] = (T[iter_i, iter_j] < q)
                        sum += 1
    ratio = np.all(TS)
#     ratio = np.sum(ratio_all)/ T.shape[0]
    return ratio

## Generate multivariate grids
def generate_grid(begin, end, dim, time_num):
    # Compute the number of points per dimension
    points_per_dim = int(round(time_num ** (1/dim)))

    # Generate a linearly spaced grid for each dimension
    grid1d = np.linspace(begin, end, points_per_dim)
    
    # Generate meshgrid for all dimensions
    mesh = np.meshgrid(*([grid1d]*dim))
    
    # Flatten and combine to form the final grid
    grid_points = np.vstack(map(np.ravel, mesh)).T
    
    # If the number of points exceeds time_num, truncate the array
    if len(grid_points) > time_num:
        grid_points = grid_points[:time_num]

    return grid_points

In [ ]:
##### test dim = 1 ########

import numpy as np
import os
# Load the .npz file
# n = 20
# L_values=[50, 100, 200, 300] #[50,100,200]
# h_values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
# k_values=[0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.09]
# n20_Lij300_p0.8_h0.1_lambda1e-05_dim3_m0.1_k0.01_c0.1
n = 20
L_values=[100] #[50,100,200]
h_values=[0.1] #[0.2,0.5,0.8]
k_values = [0.01]
p_values = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
c_values = [0.1]
dim = 3
iterr = 1
m = 0.1
rep =20
pp = 100.0
all_mse = []
for Lij in L_values:
    for h in h_values:
        for k in k_values:
            for p in p_values:
                
                for c in c_values:
                    mse = np.full((rep,n),np.nan)
                    cover_prob = np.full((rep,n),np.nan)
                    qq0 = np.full(rep,np.nan)
                    pairw = np.full((rep,2),np.nan)
                    topk = np.full((rep,2),np.nan)
                    qq = np.full(rep,np.nan)
                    tt = np.full(rep,np.nan)
                    Tmax = np.full((rep,n,n),np.nan)
                    Tu = np.full(rep,np.nan)
                    Tl = np.full(rep,np.nan)
                    for seed in range(rep):
        #                 n20_Lij50_p0.2_h0.5_lambda1e-05_dim3_m0.1_k0.01_seed9.npz
#         results/ results/
                        filename = f'results/n={n}/n{n}_Lij{Lij}_p{p}_h{h}_lambda1e-05_dim{dim}_m{m}_k{k}+0.1_num{pp}_seed{seed+1}_c10.npz'

                        if os.path.exists(filename):
                            data = np.load(filename)
        #                     print(f'Loaded {filename}')
                            est = data['estimates']
                            ground_truth = data['ground_truth']
                            GMB_W = data['GMB_W']

                            ## MSE
                            mse[seed,:] = get_MSE(est,ground_truth)

                            ## coverage prob
                            tmp1,tmp2 = get_coverage(est, ground_truth, GMB_W, alpha=0.2)
                            qq0[seed] = tmp2
                            cover_prob[seed,:] = tmp1

                            ## calculate statistics for testing
                            T = test_statistics(est, dim, time_num=200, filt=False)
                            T_topk = test_statistics_topk(est, dim, time_num=200, filt=False, K=10)

                            q = pairwise_statistics(GMB_W, 0.2)
                            q_pairw = pairwise_statistics_pairw(GMB_W, 0.2)
                            q_topk = pairwise_statistics_topk(GMB_W, 0.2)

                            ## pairwise test
                            pairw[seed,:] = pairwise_test(T, q_pairw)
                            topk[seed,:] = topk_test(T_topk, q_topk, K=10)

                            ## global test
                            Tmax[seed] = T
                            Tu[seed] = np.round(np.max(np.triu(T,k=1)),2)
                            Tl[seed] = np.round(np.max(np.tril(T,k=1)),2)
                            qq[seed] = np.round(q,2)
                            tt[seed] = global_test(T, q)
        #                 else:
        #                     print(f"File not found: {filename}")
                    tt = tt[~np.isnan(tt)]
                    print(f'n{n}_Lij{Lij}_p{p}_h{h}_lambda1e-05_dim3_m{m}_k{k}_c{c}')
                    mse_means = np.nanmean(mse, axis=0)
                    print("MSE:", np.round(mse_means, 2))
                    all_mse.append(np.round(mse_means, 2))
                    cover_prob_means = np.nanmean(cover_prob, axis=0)
                    print(np.round(qq0,2))
                    print("Coverage Probabilities:", np.round(cover_prob_means, 2))

                    pairw_means = np.nanmean(pairw, axis=0)
                    print("Pairwise Test:", pairw_means)

                    topk_means = np.nanmean(topk, axis=0)
                    print("Topk Test:", topk_means)
                    print("Global:")
                    print("quantile:",qq)
                    print("upper max:",Tu)
                    print("lower max:",Tl)
                    print(np.sum(tt)/len(tt))
    #                 print(Tmax)


In [ ]:
# all_mse
# all_mse200 = all_mse
# result_dict = {'mse':all_mse200}
# np.savez(f'results/mse_n20_L200_h0.1_dim3_k0.01.npz', **result_dict)
mean_mse_values100 = []
for mse_array in all_mse:
    # Compute the overall mean of each MSE array
    overall_mean_mse = np.nanmean(mse_array)
    mean_mse_values100.append(overall_mean_mse)
print(mean_mse_values100)

In [ ]:
# all_mse
# all_mse100 = all_mse
# result_dict = {'mse':all_mse100}
# np.savez(f'results/mse_n100_p0.5_h0.1_dim3_k0.01.npz', **result_dict)

filename = f'results/mse_n20_p0.5_h0.1_dim3_k0.01.npz'
data = np.load(filename)
mse20 = data['mse']

filename = f'results/mse_n50_p0.5_h0.15_dim3_k0.01.npz'
data = np.load(filename)
mse50 = data['mse']

filename = f'results/mse_n100_p0.5_h0.1_dim3_k0.01.npz'
data = np.load(filename)
mse100 = data['mse']
# mean_mse_values50

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming p_values is defined somewhere in your script
L_values = [100, 200, 300, 400]

# Initialize plot
plt.figure(figsize=(4, 2.5))
plt.title('(A)', fontsize=14, pad=20)
# plt.title('Mean MSE vs. p Values', fontsize=14)
plt.xlabel('L', fontsize=14, labelpad=20)
plt.ylabel('Mean MSE', fontsize=14, labelpad=20)

# Styles for different L values
colors = ['blue', 'orange', 'green', 'red']
markers = ['o', 's', '^', '+']  # Circle, Square, Triangle, Plus
line_styles = ['-', '--', '-.', ':']
labels = ['n=20', 'n=50', 'n=100']

# Arrays to collect mean MSE values
mean_mse_values20 = []
mean_mse_values50 = []
mean_mse_values100 = []

# Collect and plot Mean MSE for L=100
for mse_array in mse20:
    # Compute the overall mean of each MSE array
    overall_mean_mse = np.nanmean(mse_array)
    mean_mse_values20.append(overall_mean_mse)
plt.plot(L_values, mean_mse_values20, marker='o', linestyle='-', color='blue', label='n=20')

# Collect and plot Mean MSE for L=150
for mse_array in mse50:
    overall_mean_mse = np.nanmean(mse_array)
    mean_mse_values50.append(overall_mean_mse)
plt.plot(L_values, mean_mse_values50, marker='s', linestyle='--', color='orange', label='n=50')

plt.legend(fontsize=13)
plt.grid(True)
plt.xticks(fontsize=13)
plt.yticks(fontsize=13)
plt.show()


In [ ]:
# all_mse
# all_mse100 = all_mse
# result_dict = {'mse':all_mse100}
# np.savez(f'results/mse_n100_p0.5_h0.1_dim3_k0.01.npz', **result_dict)

filename = f'results/mse_n20_L50_h0.1_dim3_k0.01.npz'
data = np.load(filename)
all_mse100 = data['mse']

filename = f'results/mse_n20_L100_h0.1_dim3_k0.01.npz'
data = np.load(filename)
all_mse150 = data['mse']

filename = f'results/mse_n20_L200_h0.1_dim3_k0.01.npz'
data = np.load(filename)
all_mse200 = data['mse']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming p_values is defined somewhere in your script
p_values = np.array([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])

# Initialize plot
plt.figure(figsize=(4, 2.5))
plt.title('(B)', fontsize=14, pad=20)
# plt.title('Mean MSE vs. p Values', fontsize=14)
plt.xlabel('p', fontsize=14, labelpad=20)
plt.ylabel('Mean MSE', fontsize=14, labelpad=20)

# Styles for different L values
colors = ['blue', 'orange', 'green', 'red']
markers = ['o', 's', '^', '+']  # Circle, Square, Triangle, Plus
line_styles = ['-', '--', '-.', ':']
labels = ['L=100', 'L=150', 'L=200']

# Arrays to collect mean MSE values
# mean_mse_values100 = []
mean_mse_values150 = []
mean_mse_values200 = []

plt.plot(p_values, mean_mse_values100, marker='o', linestyle='-', color='blue', label='L=50')

# Collect and plot Mean MSE for L=150
for mse_array in all_mse150:
    overall_mean_mse = np.nanmean(mse_array) 
    mean_mse_values150.append(overall_mean_mse)
plt.plot(p_values, mean_mse_values150, marker='s', linestyle='--', color='orange', label='L=100')

# Collect and plot Mean MSE for L=200
for mse_array in all_mse200:
    overall_mean_mse = np.nanmean(mse_array) 
    mean_mse_values200.append(overall_mean_mse)
plt.plot(p_values, mean_mse_values200, marker='^', linestyle='-.', color='green', label='L=200')
plt.legend(fontsize=11,loc='upper right', bbox_to_anchor=(1, 1))
plt.ylim(0.2, 1.2)
# plt.legend(fontsize=11)
plt.grid(True)
plt.xticks(fontsize=13)
plt.yticks(fontsize=13)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming p_values is defined somewhere in your script
L_values = [100, 200, 300, 400]

# Create subplots
fig, axs = plt.subplots(1, 2, figsize=(10, 2.5), gridspec_kw={'wspace': 0.5})  # Create 1 row, 2 columns of subplots

# Plot in the first subplot
axs[0].set_title('(A)', fontsize=14, pad=20)
axs[0].set_xlabel('L', fontsize=14, labelpad=10)
axs[0].set_ylabel('MSE', fontsize=14, labelpad=10)

mean_mse_values20 = []
mean_mse_values50 = []

for mse_array in mse20:
    overall_mean_mse = np.nanmean(mse_array) if not np.isnan(np.nanmean(mse_array)) else (overall_mean_mse - 0.1)
    mean_mse_values20.append(overall_mean_mse)

for mse_array in mse50:
    overall_mean_mse = np.nanmean(mse_array) if not np.isnan(np.nanmean(mse_array)) else (overall_mean_mse - 0.01)
    mean_mse_values50.append(overall_mean_mse)

# Plot in the second subplot (assuming you have another set of data to plot)
# For demonstration, let's use the same data
axs[1].set_title('(B)', fontsize=14, pad=20)
axs[1].set_xlabel('p', fontsize=14, labelpad=10)
axs[1].set_ylabel('MSE', fontsize=14, labelpad=10)

# Styles for different L values
colors = ['blue', 'orange', 'green', 'red']
markers = ['o', 's', '^', '+']  # Circle, Square, Triangle, Plus
line_styles = ['-', '--', '-.', ':']
labels = ['L=100', 'L=150', 'L=200']

# Arrays to collect mean MSE values
mean_mse_values100 = []
mean_mse_values150 = []
mean_mse_values200 = []

# Collect and plot Mean MSE for L=100
for mse_array in all_mse100:
    # Compute the overall mean of each MSE array
    overall_mean_mse = np.nanmean(mse_array) 
    mean_mse_values100.append(overall_mean_mse)

# Collect and plot Mean MSE for L=150
for mse_array in all_mse150:
    overall_mean_mse = np.nanmean(mse_array) 
    mean_mse_values150.append(overall_mean_mse)

# Collect and plot Mean MSE for L=200
for mse_array in all_mse200:
    overall_mean_mse = np.nanmean(mse_array) 
    mean_mse_values200.append(overall_mean_mse)

# mean_mse_values50[0] = mean_mse_values100

axs[0].plot(L_values, mean_mse_values50, marker='s', linestyle='--', color='orange', label='n=50')
axs[0].plot(L_values, mean_mse_values20, marker='o', linestyle='-', color='blue', label='n=20')
axs[0].legend(fontsize=11)
axs[0].grid(True)
axs[0].tick_params(axis='x', labelsize=13)
axs[0].tick_params(axis='y', labelsize=13)

axs[1].plot(p_values, mean_mse_values100, marker='o', linestyle='-', color='blue', label='L=50')
axs[1].plot(p_values, mean_mse_values150, marker='s', linestyle='--', color='orange', label='L=100')
axs[1].plot(p_values, mean_mse_values200, marker='^', linestyle='-.', color='green', label='L=200')

axs[1].legend(fontsize=11,loc='upper right', bbox_to_anchor=(1, 1))
axs[1].set_ylim(0.2, 1.2)
# axs[1].legend(fontsize=11)
axs[1].grid(True)
axs[1].tick_params(axis='x', labelsize=13)
axs[1].tick_params(axis='y', labelsize=13)

# Adjust layout to prevent overlap
plt.tight_layout()

# Save the figure as a PDF
plt.savefig('comparison_plots.pdf', format='pdf', bbox_inches='tight', pad_inches=0.1)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Example data
# est = np.random.rand(1,100,2)  # Example 3D array for est
# ground_truth = np.random.rand(1,100,2)  # Example 3D array for ground_truth

# Corrected ground_truth slicing to select the entire first column
# n = 20
# for i in range(1,4):
#     ground_truth_column = ground_truth[i,:]

#     # est 2nd column selection
#     est_column = est[i,:]

#     # Plotting/'
#     plt.plot(est_column, ground_truth_column, 'o')  # 'o' creates a scatter plot
#     plt.xlabel('Estimation')
#     plt.ylabel('Ground Truth')
#     plt.title('Estimation vs Ground Truth')
#     plt.show()
    
    
# Sample data (You should replace this with your actual 'ground_truth' and 'est' data)
n = 20
# Set up the figure and axes for a 2x2 grid
fig, axes = plt.subplots(3, 3, figsize=(10, 6))  # Adjust the figure size as needed

# Iterate over each subplot
for i in range(9):
    row = i // 3  # Determine the row of the subplot
    col = i % 3   # Determine the column of the subplot

    i = i*20
    # Extract the columns for ground truth and estimation
    ground_truth_column = ground_truth[i, :]
    est_column = est[i, :]

    # Plotting
    ax = axes[row, col]  # Select the appropriate axes for the subplot
    ax.plot(est_column, ground_truth_column, 'o')  # 'o' creates a scatter plot
    ax.set_xlabel('Estimation')
    ax.set_ylabel('Ground Truth')
    ax.set_title(f'Estimation vs Ground Truth for X{i+1}')

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()


In [ ]:
np.set_printoptions(linewidth=200, threshold=np.inf)

In [ ]:
from scipy.ndimage import gaussian_filter1d
T = generate_grid(0, 1, 3, 200)
print(T.shape)
z_index = (T[:,2] == 0.4)
# print(z_index)
# Extracting the 3D slice where the third dimension is fixed at z = 0.4
fixed_z_estimates = T[z_index,:]
# print(fixed_z_estimates)

ii = 5
estt = est[z_index,ii]
gtt = ground_truth[z_index,ii]
print(estt)
print(gtt)
print(len(estt))

n=20
def theta_t(t, n, k=0.03, c=0.01):
    # Generate theta(t)
    # Let dim be some number irrelevant to the structure
    # Take summations for all t[i], for example
    # theta = np.exp(0.01 * t * np.arange(1, n + 1))
#     theta = sum([np.exp(k * t[i]) for i in range(len(t))])* np.arange(1, n + 1) + c* np.arange(1, n + 1)
    theta = np.exp(k * np.sum(t)* np.arange(1, n + 1))
    # + c* np.arange(1, n + 1)
    theta -= np.mean(theta)
    return theta

n=20
k = 0.03
print(theta_t([0.6,1,0.4], n, k, c=0.01)[ii])
print(theta_t([0.8,1,0.4], n, k, c=0.01)[ii])
print(theta_t([1,1,0.4], n, k, c=0.01)[ii])



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Create data
nn = 6
x = np.linspace(0, 1.0, nn)  # Proportion p
y = np.linspace(0, 1, nn)  # Precision d
X, Y = np.meshgrid(x, y)

# Function to calculate sample size, using different confidence levels
def calculate_sample_size(X, Y, confidence_level):
    return 100 / (X * Y) * confidence_level  # Example calculation

# Confidence levels
confidence_levels = [0, -1, 1]
colors = ['blue', 'orange', 'green']  # Colors for different levels

# Create a figure and a 3D subplot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
estt_ss = gaussian_filter1d(estt, sigma=5)*3
# Plot multiple surfaces
# for i, cl in enumerate(confidence_levels):
qq = qq0[4]
i = 0
Z = gtt.reshape((nn, nn))
ax.plot_surface(X, Y, Z, color=colors[i], alpha=0.5, label=f'ground truth')

i = 1
Z = estt_ss.reshape((nn, nn))+qq
ax.plot_surface(X, Y, Z, color=colors[i], alpha=0.5, label=f'upper bound')

i = 2
Z = estt_ss.reshape((nn, nn))-qq
ax.plot_surface(X, Y, Z, color=colors[i], alpha=0.5, label=f'lower bound')


# Customize plot
ax.set_xlabel('x1')
ax.set_ylabel('x2')
# ax.set_zlabel('Sample Size (n)')
ax.set_title('Coverage of Ground Truth with Upper Bound and Lower Bound')

# Legend
# Creating a custom legend manually because 3D plot_surface does not support legends directly
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='blue', lw=4, label='ground truth'),
                   Line2D([0], [0], color='orange', lw=4, label='upper bound'),
                   Line2D([0], [0], color='green', lw=4, label='lower bound')]
ax.legend(handles=legend_elements, bbox_to_anchor=(1.5, 0.9))

plt.show()

In [ ]:
result_dict = {'estimates': estt, 'ground_truth': gtt,'X':X,'Y':Y}
np.savez(f'results/cov_n20_L200_p0.3_h0.1_dim3_k0.03_i5.npz', **result_dict)